In [ ]:
# Ensure jacopy is importable when this notebook is opened
# directly (not via pytest). Walks up from the notebook's
# directory to the repo root and prepends it to sys.path if
# jacopy isn't already installed into this kernel.
try:
    import jacopy  # noqa: F401
except ModuleNotFoundError:
    import sys
    from pathlib import Path
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "jacopy" / "__init__.py").is_file():
            sys.path.insert(0, str(candidate))
            break
    import jacopy  # noqa: F401

# 16, Phase 13 deep dive: the `[π, π]_SN` obstruction

Companion notebook to [16_phase_13_deep_dive.md](16_phase_13_deep_dive.md). Phase 13 closes the cyclic Poisson Jacobi sum to `[·,·]_SN(π, π)` *without* citing the seeded `poisson_jacobi` theorem, using only engine-level rewrite axioms. This walks the machinery: `LieBracketVF` atom + the four rewrite rules.

## `LieBracketVF`, Lie bracket of vector fields as an atom

`[X, Y]_VF` is an opaque `Derivation` subclass with structural identity over `(X, Y)`. Why opaque: after the operator-commutator fold, downstream Cartan rules need a single derivation, not a `X*Y − Y*X` expansion. Literal expansion is still available through `LieBracket.expand` for callers who need it.

In [ ]:
from jacopy.algebra.derivation import Derivation
from jacopy.algebra.lie_bracket_vf import lie_bracket_vf

X = Derivation("X", 0)
Y = Derivation("Y", 0)

bracket = lie_bracket_vf(X, Y)
print(f"name      : {bracket}")
print(f"X         : {bracket.X}")
print(f"Y         : {bracket.Y}")
print(f"degree    : {bracket._degree}")
print(f"same atom : {bracket == lie_bracket_vf(X, Y)}")

## The vector-field axioms (Faz 13.C)

`OpCommutatorVfDefinition` folds the operator commutator: `L_X(L_Y(ω)) − L_Y(L_X(ω)) → L_{[X,Y]_VF}(ω)`. Order-permissive on the Sum's children, the upstream pipeline doesn't have to canonicalise first.

In [ ]:
from jacopy.algebra.derivation import Act
from jacopy.calculus.lie_derivative import lie_derivative
from jacopy.calculus.vf_axioms import OpCommutatorVfDefinition
from jacopy.core.expr import Symbol, Sum, Neg
from jacopy.proof.expansion import ExpansionEngine

omega = Symbol("ω")
LX, LY = lie_derivative(X), lie_derivative(Y)

expr = Sum(
    Act(LX, Act(LY, omega)),
    Neg(Act(LY, Act(LX, omega))),
)

engine = ExpansionEngine([OpCommutatorVfDefinition()])
out, steps = engine.expand(expr)
print(f"input  : {expr}")
print(f"output : {out}")
print(f"rule   : {steps[0].rule}")

The match is structural, a positive `Act(L_X, Act(L_Y, ω))` paired with its sign-flipped twin `Neg(Act(L_Y, Act(L_X, ω)))`. `LieVfJacobiDefinition` is the same kind of matcher one level up: three cyclically-permuted `Act(L_{[X,[Y,Z]_VF]_VF}, ω)` terms collapse to `Integer(0)`.

## Function-side closure (2g-deep, end-to-end)

Two axioms in `jacopy.calculus.poisson_axioms`, `PoissonAsHamiltonianDefinition` (rewrites `{f, g}_π → X_f(g)` for a *pinned* `DerivedBracket`) and `HamiltonianCyclicSnFormulaDefinition` (collapses cyclic `Act(X_a, Act(X_b, c))` to `[·,·]_SN(π, π)`), close the cyclic Poisson Jacobi sum end-to-end.

In [ ]:
from jacopy.brackets.derived import DerivedBracket
from jacopy.brackets.schouten import sn
from jacopy.calculus.poisson_axioms import (
    PoissonAsHamiltonianDefinition,
    HamiltonianCyclicSnFormulaDefinition,
)
from jacopy.core.properties import Graded
from jacopy.core.registry import PropertyRegistry

reg = PropertyRegistry()
pi = Symbol("π"); reg.declare(pi, Graded(degree=1))
f = Symbol("f"); reg.declare(f, Graded(degree=-1))
g = Symbol("g"); reg.declare(g, Graded(degree=-1))
h = Symbol("h"); reg.declare(h, Graded(degree=-1))

P = DerivedBracket(sn, pi, name="[·,·]_π")
obs = P.graded_jacobi_obstruction(f, g, h, registry=reg)
print(f"input : {obs}")

In [ ]:
engine = ExpansionEngine([
    PoissonAsHamiltonianDefinition(bracket=P, bivector=pi),
    HamiltonianCyclicSnFormulaDefinition(bivector=pi),
])

result, steps = engine.expand(obs)
print(f"steps : {len(steps)}")
for i, s in enumerate(steps):
    print(f"  [{i:02d}] {s.rule}")
print(f"final : {result}")

Seven steps: six `PoissonAsHamiltonian` rewrites (each cyclic term `{f, {g, h}}` peels twice, outer then inner, into `X_f(X_g(h))`) plus one `HamiltonianCyclicSn` collapse to `−[·,·]_SN(π, π)`. The leading `Neg` is the sign carried in from `graded_jacobi_obstruction`'s shape, not a sign error.

## Form-side asymmetry (2f-deep)

The form-side chain, cyclic Koszul Jacobi on three 1-forms, closes through `SnBivectorFormulaDefinition` (Faz 13.D), but doesn't reach a clean `[·,·]_SN(π, π)` residue without extra bookkeeping. After Cartan-layer expansion of `{α, β}_K = L_{π^♯α}β − L_{π^♯β}α − d⟨π^♯α, β⟩` and operator-commutator folding, three pieces remain:

1. **Named-bracket cyclic** `Σ_cyc L_{[π^♯·, π^♯·]_VF}(·)`, what `SnBivectorFormulaDefinition` rewrites.
2. **Iterated Lie tails** `L_{π^♯·}(L_{π^♯·}(·))`, separate cancellation pass.
3. **`d⟨·, ·⟩` residues** from the Koszul third term, don't enter the SN handle directly.

(2) and (3) are the bookkeeping burden. They cancel algebraically but need user-driven simplification or extra rules. The function-side chain doesn't have this burden, the `Act(X_f, X_g(h))` shape absorbs everything into a single iterated derivation. The asymmetry is structural: 1-forms carry more Cartan-layer machinery than functions.

## When to use the deep machinery

| Goal | Use |
|---|---|
| One-step Jacobi reduction citing a seeded theorem | `PoissonBracket.prove_jacobi_reduction` (tutorial 12) |
| Step-by-step `LieBracketVF` fold transcript | This tutorial's two-axiom engine, function-side |
| Custom derived bracket without a seeded theorem | Pin a `DerivedBracket`, layer the same axioms |
| Form-side cancellation with the 3-form pairing | Faz 13.D `SnBivectorFormulaDefinition` + manual residue work |

Default workflow stays at the seeded-theorem level, `prove_jacobi_reduction` is shorter and reads as "by the Derived Bracket Theorem". The deeper machinery sits *underneath* that one-line citation, ready when the seed doesn't apply.

## Summary

* `LieBracketVF(X, Y)`, opaque `Derivation` atom, kept unexpanded for downstream Cartan rule uniformity.
* `OpCommutatorVfDefinition` folds operator commutators into `L_{[X,Y]_VF}`; `LieVfJacobiDefinition` zeros the cyclic Lie-Jacobi triple.
* `PoissonAsHamiltonianDefinition` + `HamiltonianCyclicSnFormulaDefinition` close the cyclic Poisson Jacobi sum to `[·,·]_SN(π, π)` in 7 engine steps.
* Form-side (2f-deep) needs additional bookkeeping for iterated Lie tails and `d⟨·, ·⟩` residues, a structural consequence of 1-forms' deeper Cartan layer.